<div style="background: linear-gradient(120deg, #12263A, #244A63); padding: 28px; border-radius: 14px; color: white;">
  <div style="font-size: 13px; letter-spacing: 2px; opacity: .85;">PATTERN RECOGNITION & MACHINE LEARNING · COURSE PROJECT</div>
  <h1 style="color: white; margin: 12px 0 6px 0;">Intelligent Travel Mode Choice Prediction</h1>
  <div style="font-size: 16px; opacity: .92;">A classical machine-learning study using the London Passenger Mode Choice (LPMC) dataset</div>
  <div style="margin-top: 20px; font-size: 13px; opacity: .8;">Gati Shakti Vishwavidyalaya · B.Tech AI & DS · PRML</div>
</div>

### Notebook roadmap

This notebook follows the project requirements from the whiteboard and the PRML laboratory plan. Each section explains **what we are doing, why it is needed, and what to look for** before showing the code.

| Stage | Work |
|---|---|
| 01 | Problem definition, scope and dataset loading |
| 02 | Data inspection, cleaning and exploratory data analysis |
| 03 | Feature engineering and deliberate feature dropping |
| 04 | Preprocessing and train/test preparation |
| 05 | Supervised classification with syllabus-level algorithms |
| 06 | Feature selection and PCA-based model development |
| 07 | K-Means and DBSCAN clustering |
| 08 | Hard voting, soft voting and final evaluation |
| 09 | Findings, limitations and project conclusion |

> **Keep the scope honest:** this project predicts a person's choice among four urban travel modes. It does not identify GPS movement, predict flight/train/bus as separate target classes, or use weather/elevation features that are not in LPMC.

## 01 · Problem definition and project scope

### Research question

**Can a classical machine-learning model predict the travel mode selected for a journey from trip, passenger and alternative-route attributes?**

The target variable is `travel_mode`, with four classes:

| Target class | Meaning |
|---|---|
| Walking | The recorded trip was made by walking |
| Cycling | The recorded trip was made by bicycle |
| Public transport | A public-transport journey; the route may contain bus and/or rail legs |
| Driving | Driving / road-vehicle category used by the dataset |

**Important interpretation:** bus and rail durations are separate input features describing the available public-transport route. They are **not separate target classes**. Flights are not represented in this urban London dataset. Mixed-mode trips are assigned to the mode used for the greatest distance in the original study.

### What the assignment asks us to demonstrate

- A tabular dataset with more than 5,000 observations and at least 10 candidate input features.
- Exploratory data analysis and multiclass classification.
- A complete supervised-learning pipeline.
- Unsupervised learning using K-Means (choose a useful value of *k*) and DBSCAN.
- Baseline models using all eligible features, followed by feature selection and feature dropping.
- A separate PCA-based model experiment.
- Stratified K-fold evaluation.
- Ensemble classification using hard voting and soft voting.

### Course alignment

The selected methods stay within the supplied PRML course outline: preprocessing/EDA; logistic regression; decision trees; random forests; SVM; KNN; clustering with K-Means and DBSCAN; PCA; and feature-selection methods. We do not use deep-learning frameworks, XGBoost, imbalanced-learn, external boosting packages, or AutoML tools.

## 02 · Dataset information and source

**Dataset:** London Passenger Mode Choice (LPMC), based on London Travel Demand Survey journeys from April 2012 to March 2015.

- Published dataset size: **81,086 journeys**.
- Target: four travel modes — walking, cycling, public transport and driving.
- Original class proportions are approximately 17.6%, 3.0%, 35.3% and 44.2%, respectively. The exact counts will be calculated from the downloaded file below.
- The dataset contains passenger/demographic information and attributes of the available travel alternatives, such as their estimated durations and costs.

**Download:** [Official EPFL/Biogeme LPMC page](https://transp-or.epfl.ch/biogeme-3.3.2/) · [LPMC data file](https://transp-or.epfl.ch/data/lpmc.dat) · [Original paper](https://doi.org/10.1680/jsmic.17.00018)

The original paper asks researchers using the dataset to cite Hillel, Elshafie and Jin (2018): *Recreating passenger mode choice-sets for transport simulation: A case study of London, UK*.

### Data dictionary — expected original columns

The commonly distributed tabular file has 32 columns: 4 identifiers, 1 target and 27 candidate predictors. The code below checks the actual file and prints the columns it finds. If you use a differently preprocessed copy, names may vary; do not silently assume the same schema.

| Group | Candidate columns | Meaning |
|---|---|---|
| Passenger / trip context | `purpose`, `fueltype`, `faretype`, `bus_scale`, `survey_year`, `travel_year`, `travel_month`, `travel_date`, `day_of_week`, `start_time`, `age`, `female`, `driving_license`, `car_ownership` | Trip purpose, fare/fuel categories, survey timing and passenger/household attributes |
| Journey distance | `distance` | Straight-line origin-to-destination distance, in metres |
| Alternative travel durations | `dur_walking`, `dur_cycling`, `dur_pt_access`, `dur_pt_rail`, `dur_pt_bus`, `dur_pt_int`, `dur_driving` | Estimated durations for walking, cycling, public transport components and driving; generally in hours |
| Public transport | `pt_interchanges`, `cost_transit` | Number of interchanges and public-transport cost |
| Driving | `cost_driving_fuel`, `cost_driving_ccharge`, `driving_traffic_percent` | Fuel cost, congestion-charge cost and traffic variability |
| Identifiers (not model inputs) | `trip_id`, `household_id`, `person_n`, `trip_n` | Record / household / person / trip identifiers |
| Target (not a model input) | `travel_mode` | Chosen travel mode |

Some research papers describe equivalent variables with alternate names (for example `cost_pt`, `n_ints`, or colon-separated duration names). The notebook normalises a short list of known aliases when possible and then validates the required target/features.

## 03 · Import the permitted libraries

The project uses only the requested packages: **pandas, NumPy, Matplotlib, Seaborn and scikit-learn**. `pathlib` is part of Python's standard library and is used only to check whether the downloaded data file exists.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    silhouette_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans, DBSCAN

sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelsize"] = 11

RANDOM_STATE = 42
print("Libraries imported. Random state:", RANDOM_STATE)

## 04 · Load the LPMC data

Download `lpmc.dat` from the official data link and place it in the **same folder as this notebook**. The original file is tab-separated, so `sep="\t"` is important.

The cell first looks for a local file. If it is not present, it attempts the official URL. If internet access blocks that request, download the file manually from the link above and rerun this cell.

In [ ]:
DATA_PATH = Path("lpmc.dat")
DATA_URL = "https://transp-or.epfl.ch/data/lpmc.dat"

if DATA_PATH.exists():
    df_raw = pd.read_csv(DATA_PATH, sep="\t")
    print(f"Loaded local file: {DATA_PATH.resolve()}")
else:
    try:
        df_raw = pd.read_csv(DATA_URL, sep="\t")
        print("Loaded data from the official EPFL URL.")
    except Exception as exc:
        raise FileNotFoundError(
            "Could not find lpmc.dat locally or download it from EPFL. "
            "Download https://transp-or.epfl.ch/data/lpmc.dat, place it beside this notebook, "
            "and rerun this cell. Original error: " + str(exc)
        )

# Normalise whitespace in column names.
df_raw.columns = [str(col).strip() for col in df_raw.columns]
print("Raw shape:", df_raw.shape)
display(df_raw.head())

## 05 · Validate and standardise column names

Different published copies sometimes use slightly different spellings. This cell maps only known aliases to the names used in the rest of this notebook. It does **not** invent missing columns.

In [ ]:
COLUMN_ALIASES = {
    "fuel_type": "fueltype",
    "fare_type": "faretype",
    "cost_pt": "cost_transit",
    "n_ints": "pt_interchanges",
    "traffic_var": "driving_traffic_percent",
    "dur_pt:rail": "dur_pt_rail",
    "dur_pt:bus": "dur_pt_bus",
    "dur_pt:access": "dur_pt_access",
    "dur_pt:interchange": "dur_pt_int",
    "cost_driving:VOC": "cost_driving_fuel",
    "cost_driving:con_charge": "cost_driving_ccharge",
}

df = df_raw.rename(columns={k: v for k, v in COLUMN_ALIASES.items() if k in df_raw.columns}).copy()

# If an alternate dataset contains a sex column instead of the 0/1 female column,
# preserve it as a categorical predictor rather than guessing its numeric coding.
if "sex" in df.columns and "female" not in df.columns:
    print("Note: this file uses 'sex' rather than 'female'; 'sex' will be retained as a categorical feature.")

EXPECTED_PREDICTORS = [
    "purpose", "fueltype", "faretype", "bus_scale", "survey_year", "travel_year",
    "travel_month", "travel_date", "day_of_week", "start_time", "age", "female",
    "driving_license", "car_ownership", "distance", "dur_walking", "dur_cycling",
    "dur_pt_access", "dur_pt_rail", "dur_pt_bus", "dur_pt_int", "pt_interchanges",
    "dur_driving", "cost_transit", "cost_driving_fuel", "cost_driving_ccharge",
    "driving_traffic_percent"
]

if "sex" in df.columns and "female" not in df.columns:
    EXPECTED_PREDICTORS = [c for c in EXPECTED_PREDICTORS if c != "female"] + ["sex"]

missing_target = "travel_mode" not in df.columns
available_predictors = [c for c in EXPECTED_PREDICTORS if c in df.columns]
missing_predictors = [c for c in EXPECTED_PREDICTORS if c not in df.columns]

print("Available predictors:", len(available_predictors))
print("Missing expected predictors:", missing_predictors)
print("Other columns:", [c for c in df.columns if c not in EXPECTED_PREDICTORS and c != "travel_mode"])

if missing_target:
    raise ValueError("The file does not contain the required target column 'travel_mode'. Check that you downloaded the original LPMC data file.")
if len(available_predictors) < 10:
    raise ValueError("Fewer than 10 expected predictor columns were found. Check the dataset version and column names before proceeding.")

## 06 · First inspection: rows, columns, types and missing values

Before modelling, inspect the actual file. Confirm that it exceeds the assignment's minimum of 5,000 observations and 10 input features. Also check data types, duplicate rows and missing values.

In [ ]:
print(f"Observations: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Candidate predictors found: {len(available_predictors)}")
print("Exact duplicate rows:", int(df.duplicated().sum()))

display(df.dtypes.rename("dtype").to_frame())
display(df.isna().sum().sort_values(ascending=False).rename("missing_values").to_frame().head(15))

assert len(df) > 5000, "The loaded dataset does not meet the >5,000-sample requirement."
assert len(available_predictors) >= 10, "The loaded dataset does not meet the 10-feature requirement."

## 07 · Clean exact duplicate rows and prepare the target label

We remove exact duplicate rows and rows with a missing target. We **do not** delete rows merely because one input feature is missing; missing predictor values will be handled inside the model pipeline using median or most-frequent imputation.

The standard LPMC file uses four integer codes. Some repackaged copies shift these codes to start at zero, so the cell checks which coding scheme is present before assigning readable class names. If the values do not match either scheme, stop and inspect the dataset rather than guessing.

In [ ]:
df = df.drop_duplicates().copy()
df = df.loc[df["travel_mode"].notna()].copy()

mode_values = set(pd.Series(df["travel_mode"]).dropna().unique().tolist())
mode_map_1_based = {1: "Walking", 2: "Cycling", 3: "Public transport", 4: "Driving"}
mode_map_0_based = {0: "Walking", 1: "Cycling", 2: "Public transport", 3: "Driving"}

if mode_values.issubset(set(mode_map_1_based)) and 4 in mode_values:
    y = df["travel_mode"].map(mode_map_1_based)
    print("Detected 1-based LPMC target coding (1 to 4).")
elif mode_values.issubset(set(mode_map_0_based)) and 0 in mode_values:
    y = df["travel_mode"].map(mode_map_0_based)
    print("Detected zero-based target coding (0 to 3).")
else:
    raise ValueError(f"Unexpected travel_mode values: {sorted(mode_values)}. Inspect the data and verify its label encoding.")

if y.isna().any():
    raise ValueError("Some target labels could not be mapped. Inspect travel_mode values.")

df["travel_mode_label"] = y
CLASS_ORDER = ["Walking", "Cycling", "Public transport", "Driving"]
print("Final cleaned rows:", f"{len(df):,}")
display(df["travel_mode_label"].value_counts().reindex(CLASS_ORDER, fill_value=0).rename("count").to_frame())

## 08 · Exploratory Data Analysis (EDA): target distribution

The original study reports a naturally imbalanced target: cycling is much less frequent than driving or public transport. Plot the actual counts and percentages in our copy. This matters because a model can appear accurate by predicting majority classes too often.

In [ ]:
class_counts = df["travel_mode_label"].value_counts().reindex(CLASS_ORDER, fill_value=0)
class_percent = (class_counts / class_counts.sum() * 100).round(2)
class_summary = pd.DataFrame({"count": class_counts, "percentage": class_percent})
display(class_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(x=class_summary.index, y=class_summary["count"], ax=axes[0])
axes[0].set_title("Number of journeys per travel mode")
axes[0].set_xlabel("Travel mode")
axes[0].set_ylabel("Journeys")
axes[0].tick_params(axis="x", rotation=18)

axes[1].pie(class_summary["count"], labels=class_summary.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Travel mode share")
plt.tight_layout()
plt.show()

## 09 · EDA: numeric distributions and relationships

Use histograms and boxplots to understand scale, skew and potential outliers. The variables use different units: distance is in metres, most durations are in hours, and costs are monetary values. Scaling will therefore be required for distance-based and margin-based algorithms such as KNN, SVM and K-Means.

In [ ]:
# Identify numeric candidate predictors from the data as loaded.
eda_numeric = [c for c in available_predictors if pd.api.types.is_numeric_dtype(df[c])]
print("Numeric predictors:", eda_numeric)

plot_columns = [c for c in ["distance", "dur_walking", "dur_cycling", "dur_pt_rail", "dur_pt_bus", "dur_driving", "cost_transit", "driving_traffic_percent"] if c in df.columns]

if plot_columns:
    df[plot_columns].hist(bins=30, figsize=(14, 10), edgecolor="white")
    plt.suptitle("Distributions of selected numeric journey features", y=1.02, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()

box_columns = [c for c in ["distance", "dur_walking", "dur_cycling", "dur_driving", "cost_transit"] if c in df.columns]
if box_columns:
    fig, axes = plt.subplots(len(box_columns), 1, figsize=(12, 3.2 * len(box_columns)))
    if len(box_columns) == 1:
        axes = [axes]
    for ax, col in zip(axes, box_columns):
        sns.boxplot(data=df, x="travel_mode_label", y=col, order=CLASS_ORDER, ax=ax)
        ax.set_title(f"{col} by chosen travel mode")
        ax.tick_params(axis="x", rotation=12)
    plt.tight_layout()
    plt.show()

## 10 · EDA: correlation heatmap

Correlation helps identify pairs of numeric variables that carry overlapping information. High correlation does not automatically mean that a feature must be removed, but it gives us a reason to test a compact feature representation later.

In [ ]:
correlation_columns = [c for c in eda_numeric if df[c].nunique(dropna=True) > 1]
if len(correlation_columns) >= 2:
    corr = df[correlation_columns].corr(numeric_only=True)
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr, cmap="vlag", center=0, annot=False, linewidths=0.25)
    plt.title("Correlation heatmap — numeric candidate predictors")
    plt.tight_layout()
    plt.show()

## 11 · Define the baseline feature set and remove identifiers

The four identifier columns are useful for tracing records, but they should not be used as predictive inputs. Numeric identifiers can let a model memorise arbitrary record ordering or household/person references rather than learn general travel-choice patterns.

We keep `household_id` separately for a later discussion of repeated observations, but remove it from `X`. We also exclude the target from the predictors. The first supervised experiment uses **all available candidate predictors** (normally 27).

In [ ]:
ID_COLUMNS = [c for c in ["trip_id", "household_id", "person_n", "trip_n", "ori_postcode", "desti_postcode"] if c in df.columns]
GROUP_ID = df["household_id"].copy() if "household_id" in df.columns else None

X_full = df[available_predictors].copy()
y = df["travel_mode_label"].copy()

print("Identifier columns excluded from X:", ID_COLUMNS)
print("Baseline predictor count:", X_full.shape[1])
print("Baseline matrix shape:", X_full.shape)
print("Target shape:", y.shape)

## 12 · Feature engineering and deliberate feature dropping

This section is included because the project brief asks us to investigate feature dropping, not merely train one model and stop.

We create a **compact alternative representation**:

1. Combine the four public-transport duration components into `dur_pt_total`.
2. Combine fuel and congestion-charge costs into `cost_driving_total`.
3. Drop the component columns after creating those totals.

The baseline experiment still retains the original components. The compact experiment tests whether fewer, more aggregated inputs can provide comparable performance. We do not include a total and all of its components in the same compact feature set because that would duplicate information.

In [ ]:
X_compact = X_full.copy()
pt_parts = [c for c in ["dur_pt_access", "dur_pt_rail", "dur_pt_bus", "dur_pt_int"] if c in X_compact.columns]
driving_cost_parts = [c for c in ["cost_driving_fuel", "cost_driving_ccharge"] if c in X_compact.columns]

if len(pt_parts) == 4:
    X_compact["dur_pt_total"] = X_compact[pt_parts].sum(axis=1, min_count=4)
    X_compact = X_compact.drop(columns=pt_parts)
    print("Created dur_pt_total and dropped:", pt_parts)

if len(driving_cost_parts) == 2:
    X_compact["cost_driving_total"] = X_compact[driving_cost_parts].sum(axis=1, min_count=2)
    X_compact = X_compact.drop(columns=driving_cost_parts)
    print("Created cost_driving_total and dropped:", driving_cost_parts)

print("Full feature count:", X_full.shape[1])
print("Compact feature count:", X_compact.shape[1])
print("Dropped/aggregated components:", sorted(set(X_full.columns) - set(X_compact.columns)))

## 13 · Train/test split

We reserve 20% of the records as a final hold-out test set and preserve the class proportions with stratification. The test set is not used to fit the preprocessing steps, select features or tune models.

**Data-structure caution:** LPMC contains repeated trips from the same people and households. A random stratified split can place journeys from one household in both train and test sets, so the reported score may be more optimistic than a strict household-held-out evaluation. For this course notebook we use the requested stratified split and will state this limitation in the conclusion. A grouped split can be added as a robustness check if your instructor asks for it.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_full,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training rows:", f"{len(X_train):,}")
print("Testing rows:", f"{len(X_test):,}")
print("Train class proportions:")
display(y_train.value_counts(normalize=True).reindex(CLASS_ORDER).rename("proportion").to_frame())
print("Test class proportions:")
display(y_test.value_counts(normalize=True).reindex(CLASS_ORDER).rename("proportion").to_frame())

## 14 · Build a reusable preprocessing pipeline

Different algorithms need different input preparation. Numeric columns are median-imputed and standardised. Categorical columns are filled with their most frequent value and one-hot encoded. This keeps nominal categories from being treated as ordered numbers.

The `ColumnTransformer` and `Pipeline` ensure preprocessing is fitted only on the training portion of each cross-validation fold. That prevents information from validation/test folds leaking into training.

In [ ]:
CATEGORICAL_HINTS = [
    "purpose", "fueltype", "faretype", "survey_year", "travel_year",
    "travel_month", "travel_date", "day_of_week", "car_ownership", "sex"
]

def build_preprocessor(X_reference):
    categorical_columns = [c for c in CATEGORICAL_HINTS if c in X_reference.columns]
    numeric_columns = [c for c in X_reference.columns if c not in categorical_columns]

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_columns),
            ("categorical", categorical_pipeline, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )

preprocessor = build_preprocessor(X_train)
print("Numeric columns:", [c for c in X_train.columns if c not in CATEGORICAL_HINTS])
print("Categorical columns:", [c for c in CATEGORICAL_HINTS if c in X_train.columns])

## 15 · Baseline supervised models using all features

We now compare familiar classifiers from the PRML laboratory plan:

- **Logistic Regression** — linear multiclass baseline.
- **Decision Tree** — rule-based classifier; we restrict depth to reduce overfitting.
- **Random Forest** — ensemble of decision trees.
- **Linear SVM** — a support-vector classifier with a linear decision boundary.
- **K-Nearest Neighbours (KNN)** — an instance-based classifier.

To keep the five-fold experiment practical on a laptop, cross-validation is run on a **stratified training subset** (up to 20,000 rows). The final selected model is then refitted on the full training split and evaluated once on the untouched test split. This sampling choice is printed and must be disclosed in the report.

In [ ]:
model_definitions = {
    "Logistic Regression": LogisticRegression(max_iter=1500, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=14, min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=160, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
    "Linear SVM": LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, max_iter=5000),
    "KNN": KNeighborsClassifier(n_neighbors=9, weights="distance", n_jobs=-1),
}

# A stratified sample keeps the model-comparison stage manageable.
CV_SAMPLE_SIZE = min(20000, len(X_train))
if CV_SAMPLE_SIZE < len(X_train):
    X_cv, _, y_cv, _ = train_test_split(
        X_train, y_train,
        train_size=CV_SAMPLE_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_train,
    )
else:
    X_cv, y_cv = X_train.copy(), y_train.copy()

print(f"Five-fold CV sample: {len(X_cv):,} rows out of {len(X_train):,} training rows.")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "balanced_accuracy": "balanced_accuracy", "f1_macro": "f1_macro"}
cv_rows = []

for model_name, classifier in model_definitions.items():
    model_pipeline = Pipeline(steps=[
        ("preprocess", build_preprocessor(X_cv)),
        ("classifier", classifier),
    ])
    scores = cross_validate(
        model_pipeline,
        X_cv,
        y_cv,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )
    cv_rows.append({
        "Model": model_name,
        "Mean accuracy": scores["test_accuracy"].mean(),
        "Mean balanced accuracy": scores["test_balanced_accuracy"].mean(),
        "Mean macro F1": scores["test_f1_macro"].mean(),
        "Macro F1 std": scores["test_f1_macro"].std(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values("Mean macro F1", ascending=False).reset_index(drop=True)
display(cv_results.style.format({
    "Mean accuracy": "{:.3f}",
    "Mean balanced accuracy": "{:.3f}",
    "Mean macro F1": "{:.3f}",
    "Macro F1 std": "{:.3f}",
}))

## 16 · Fit the leading baseline model and evaluate the hold-out test set

Macro F1 gives each class equal weight, so the small cycling class is not hidden by the larger driving and public-transport classes. We select the leading model from cross-validation, fit it on the full training set, and evaluate on the test set once.

In [ ]:
best_model_name = cv_results.loc[0, "Model"]
best_classifier = model_definitions[best_model_name]

best_pipeline = Pipeline(steps=[
    ("preprocess", build_preprocessor(X_train)),
    ("classifier", best_classifier),
])
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_test)

final_metrics = pd.DataFrame([{
    "Selected model": best_model_name,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
    "Macro precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
}])
display(final_metrics.style.format({c: "{:.3f}" for c in final_metrics.columns if c != "Selected model"}))

print(classification_report(y_test, y_pred, labels=CLASS_ORDER, zero_division=0))

## 17 · Confusion matrix

The confusion matrix shows which modes the model mixes up. Rows are actual modes and columns are predicted modes. Read the diagonal as correct predictions; off-diagonal cells show misclassifications.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=CLASS_ORDER)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER)
plt.title(f"Confusion matrix — {best_model_name}")
plt.xlabel("Predicted mode")
plt.ylabel("Actual mode")
plt.tight_layout()
plt.show()

## 18 · Feature importance and further feature dropping

This is a second, explicit feature-reduction experiment. We fit a Random Forest on the original 27-feature representation, inspect its impurity-based feature importances, and aggregate one-hot encoded levels back to their original raw column names. This makes the result easier to explain.

**Caution:** Random-Forest impurity importance can favour variables with many possible split points and can distribute importance across correlated features. Treat it as a guide for an experiment, not proof of causation. The final feature set is tested with cross-validation rather than selected from the hold-out test results.

In [ ]:
rf_for_importance = Pipeline(steps=[
    ("preprocess", build_preprocessor(X_train)),
    ("classifier", RandomForestClassifier(
        n_estimators=160,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])
rf_for_importance.fit(X_train, y_train)

transformed_names = rf_for_importance.named_steps["preprocess"].get_feature_names_out()
importances = rf_for_importance.named_steps["classifier"].feature_importances_
importance_table = pd.DataFrame({"encoded_feature": transformed_names, "importance": importances})

# Map one-hot encoded columns back to their source feature by matching the prefix.
def source_feature(encoded_name, raw_columns):
    matches = [col for col in raw_columns if encoded_name == col or encoded_name.startswith(col + "_")]
    return max(matches, key=len) if matches else encoded_name

importance_table["source_feature"] = importance_table["encoded_feature"].apply(lambda name: source_feature(name, list(X_train.columns)))
raw_importance = importance_table.groupby("source_feature")["importance"].sum().sort_values(ascending=False)
display(raw_importance.rename("aggregated_importance").to_frame())

plt.figure(figsize=(10, 7))
raw_importance.sort_values().plot(kind="barh")
plt.title("Random Forest importance aggregated to original features")
plt.xlabel("Summed feature importance")
plt.tight_layout()
plt.show()

### 18.1 Compare the full feature set with the compact feature set

The compact set uses the two engineered totals and drops their component columns. We compare Logistic Regression with the same stratified folds and the same cross-validation sample. This directly measures whether feature dropping/aggregation keeps useful predictive information.

In [ ]:
X_compact_train = X_compact.loc[X_train.index]
X_compact_cv = X_compact_train.loc[X_cv.index]

compact_pipeline = Pipeline(steps=[
    ("preprocess", build_preprocessor(X_compact_cv)),
    ("classifier", LogisticRegression(max_iter=1500, class_weight="balanced")),
])
full_lr_pipeline = Pipeline(steps=[
    ("preprocess", build_preprocessor(X_cv)),
    ("classifier", LogisticRegression(max_iter=1500, class_weight="balanced")),
])

full_lr_scores = cross_validate(full_lr_pipeline, X_cv, y_cv, cv=cv, scoring="f1_macro", n_jobs=1)
compact_lr_scores = cross_validate(compact_pipeline, X_compact_cv, y_cv, cv=cv, scoring="f1_macro", n_jobs=1)

drop_comparison = pd.DataFrame({
    "Feature representation": ["All original candidate features", "Compact: aggregated durations/costs"],
    "Input feature count": [X_cv.shape[1], X_compact_cv.shape[1]],
    "Mean CV macro F1": [full_lr_scores["test_score"].mean(), compact_lr_scores["test_score"].mean()],
    "CV macro F1 std": [full_lr_scores["test_score"].std(), compact_lr_scores["test_score"].std()],
})
display(drop_comparison.style.format({"Mean CV macro F1": "{:.3f}", "CV macro F1 std": "{:.3f}"}))

### 18.2 Select the top *k* transformed features with scikit-learn

`SelectKBest` with the ANOVA F-test is a course-level feature-selection method available in scikit-learn. It ranks the transformed numerical and one-hot features by their univariate relationship with the target. We keep 15 transformed features for this experiment; this is a comparison setting, not a claim that 15 is universally optimal.

In [ ]:
K_BEST = min(15, len(transformed_names))
selected_pipeline = Pipeline(steps=[
    ("preprocess", build_preprocessor(X_cv)),
    ("select", SelectKBest(score_func=f_classif, k=K_BEST)),
    ("classifier", LogisticRegression(max_iter=1500, class_weight="balanced")),
])
selected_scores = cross_validate(selected_pipeline, X_cv, y_cv, cv=cv, scoring="f1_macro", n_jobs=1)

# Fit once on the CV sample to display which transformed features were selected.
selected_pipeline.fit(X_cv, y_cv)
selector = selected_pipeline.named_steps["select"]
selected_transformed_names = selected_pipeline.named_steps["preprocess"].get_feature_names_out()
selected_names = selected_transformed_names[selector.get_support()]
print(f"Selected {len(selected_names)} of {len(transformed_names)} transformed features:")
for name in selected_names:
    print(" •", name)

selection_comparison = pd.DataFrame({
    "Experiment": ["All transformed features", f"SelectKBest top {K_BEST}"],
    "Mean CV macro F1": [full_lr_scores["test_score"].mean(), selected_scores["test_score"].mean()],
})
display(selection_comparison.style.format({"Mean CV macro F1": "{:.3f}"}))

## 19 · PCA-based model development

PCA is an unsupervised dimensionality-reduction method. It transforms correlated input variables into a smaller set of principal components that retain as much variance as possible. Here we standardise and one-hot encode the inputs first, then retain enough components to explain **95% of the variance**.

We train Logistic Regression on the PCA representation and compare its cross-validation macro F1 with the full-feature Logistic Regression baseline. PCA components are mathematical combinations of original columns, so they are less directly interpretable.

In [ ]:
pca_pipeline = Pipeline(steps=[
    ("preprocess", build_preprocessor(X_cv)),
    ("pca", PCA(n_components=0.95, svd_solver="full")),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced")),
])
pca_scores = cross_validate(pca_pipeline, X_cv, y_cv, cv=cv, scoring="f1_macro", n_jobs=1)

pca_pipeline.fit(X_cv, y_cv)
pca_step = pca_pipeline.named_steps["pca"]
print("Original transformed feature count:", len(transformed_names))
print("PCA component count for 95% variance:", pca_step.n_components_)
print("Cumulative explained variance:", round(pca_step.explained_variance_ratio_.sum(), 4))

pca_comparison = pd.DataFrame({
    "Representation": ["All transformed features", "PCA (95% variance)"],
    "Mean CV macro F1": [full_lr_scores["test_score"].mean(), pca_scores["test_score"].mean()],
})
display(pca_comparison.style.format({"Mean CV macro F1": "{:.3f}"}))

## 20 · Unsupervised learning: prepare inputs for clustering

Clustering does not use the target labels. Here we use a focused group of numeric journey/alternative attributes so the experiment has a clear transport interpretation. Values are standardised because K-Means and DBSCAN rely on distances.

To keep computation reasonable, the clustering algorithms run on a random sample of at most 5,000 trips. This is a computational subset for the unsupervised experiment; the full labelled dataset remains available for supervised learning.

In [ ]:
CLUSTER_FEATURES = [
    "distance", "dur_walking", "dur_cycling", "dur_pt_access", "dur_pt_rail",
    "dur_pt_bus", "dur_pt_int", "dur_driving", "pt_interchanges",
    "cost_transit", "cost_driving_fuel", "cost_driving_ccharge", "driving_traffic_percent"
]
cluster_columns = [c for c in CLUSTER_FEATURES if c in X_full.columns]

if len(cluster_columns) < 3:
    raise ValueError("Not enough numeric transportation features for the clustering demonstration.")

cluster_source = X_full[cluster_columns].copy()
cluster_source = cluster_source.apply(pd.to_numeric, errors="coerce")
cluster_source = cluster_source.fillna(cluster_source.median())
cluster_sample = cluster_source.sample(n=min(5000, len(cluster_source)), random_state=RANDOM_STATE)

cluster_scaler = StandardScaler()
X_cluster = cluster_scaler.fit_transform(cluster_sample)
print("Clustering feature count:", len(cluster_columns))
print("Clustering sample rows:", len(cluster_sample))

## 21 · K-Means: estimate a useful value of *k*

K-Means divides observations into *k* clusters by assigning each point to its nearest cluster centre. The assignment asks us to examine different values of *k* rather than choosing one without evidence.

We use two simple diagnostics:

- **Inertia / elbow plot:** within-cluster sum of squared distances; look for a point where improvement begins to slow.
- **Silhouette score:** measures how well-separated clusters are; larger is generally better, but it should be interpreted with domain context.

Clustering is exploratory. A cluster is not automatically a true travel mode and is not evaluated as if it were a supervised prediction.

In [ ]:
k_values = range(2, 9)
inertias = []
silhouette_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_k = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster, labels_k, sample_size=min(3000, len(X_cluster)), random_state=RANDOM_STATE))

kmeans_diagnostics = pd.DataFrame({"k": list(k_values), "inertia": inertias, "silhouette": silhouette_scores})
display(kmeans_diagnostics)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set_title("K-Means elbow curve")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(k_values), silhouette_scores, marker="o")
axes[1].set_title("Silhouette score by k")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")
plt.tight_layout()
plt.show()

best_k_by_silhouette = int(kmeans_diagnostics.loc[kmeans_diagnostics["silhouette"].idxmax(), "k"])
print("Candidate k with the highest silhouette score:", best_k_by_silhouette)
print("Inspect the elbow curve too; do not choose k using one metric blindly.")

## 22 · DBSCAN: density-based clustering

DBSCAN groups points in dense regions and labels isolated points as noise (`-1`). It does not require a preset number of clusters. Its key parameters are:

- `eps`: neighbourhood radius.
- `min_samples`: number of nearby points required to form a dense region.

DBSCAN can be sensitive to scaling and to the choice of `eps`. It can also be expensive on large samples, so we use at most 3,000 standardised rows here. We test a small, understandable range of `eps` values and report the number of clusters and the noise percentage.

In [ ]:
DBSCAN_SAMPLE_SIZE = min(3000, len(X_cluster))
X_dbscan = X_cluster[:DBSCAN_SAMPLE_SIZE]

dbscan_rows = []
dbscan_labels_by_eps = {}
for eps_value in [0.6, 0.8, 1.0, 1.2, 1.5]:
    db = DBSCAN(eps=eps_value, min_samples=10)
    labels_db = db.fit_predict(X_dbscan)
    non_noise = labels_db != -1
    cluster_ids = set(labels_db[non_noise])
    n_clusters = len(cluster_ids)
    noise_percent = float((~non_noise).mean() * 100)

    if n_clusters >= 2 and non_noise.sum() > n_clusters:
        sil = silhouette_score(X_dbscan[non_noise], labels_db[non_noise], sample_size=min(1500, int(non_noise.sum())), random_state=RANDOM_STATE)
    else:
        sil = np.nan

    dbscan_rows.append({
        "eps": eps_value,
        "clusters (excluding noise)": n_clusters,
        "noise percentage": noise_percent,
        "silhouette (non-noise points)": sil,
    })
    dbscan_labels_by_eps[eps_value] = labels_db

dbscan_results = pd.DataFrame(dbscan_rows)
display(dbscan_results.style.format({
    "noise percentage": "{:.1f}%",
    "silhouette (non-noise points)": "{:.3f}",
}))

## 23 · Visualise the DBSCAN result with PCA

The next cell projects the clustering sample to two principal components **for visualisation only**. The DBSCAN labels are still the labels calculated in the standardised clustering-feature space. This plot is a 2D view of a higher-dimensional result, so some overlap can appear.

In [ ]:
visual_pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_cluster_2d = visual_pca.fit_transform(X_dbscan)

# Choose the tested eps with the largest number of clusters, then lower noise as a tie-breaker.
dbscan_choice = dbscan_results.sort_values(
    by=["clusters (excluding noise)", "noise percentage"],
    ascending=[False, True]
).iloc[0]
chosen_eps = float(dbscan_choice["eps"])
chosen_dbscan_labels = dbscan_labels_by_eps[chosen_eps]

plt.figure(figsize=(9, 6))
scatter = plt.scatter(
    X_cluster_2d[:, 0], X_cluster_2d[:, 1],
    c=chosen_dbscan_labels, cmap="tab20", s=12, alpha=0.65
)
plt.title(f"DBSCAN labels viewed in 2D PCA space (eps={chosen_eps})")
plt.xlabel("Principal component 1")
plt.ylabel("Principal component 2")
plt.tight_layout()
plt.show()
print("DBSCAN plot is an exploratory projection; it does not prove clusters correspond to travel modes.")

## 24 · Ensemble learning: hard voting and soft voting

A voting classifier combines multiple different classifiers:

- **Hard voting:** each classifier votes for one class; the class with the most votes wins.
- **Soft voting:** class probabilities are averaged; the class with the highest average probability wins.

We combine Logistic Regression, Decision Tree and Random Forest. All three support `predict_proba`, which is required for soft voting. The estimators are wrapped in their own preprocessing pipelines so every model receives correctly prepared inputs.

In [ ]:
def make_voting_estimators(X_reference):
    return [
        ("logistic", Pipeline(steps=[
            ("preprocess", build_preprocessor(X_reference)),
            ("classifier", LogisticRegression(max_iter=1500, class_weight="balanced")),
        ])),
        ("tree", Pipeline(steps=[
            ("preprocess", build_preprocessor(X_reference)),
            ("classifier", DecisionTreeClassifier(max_depth=14, min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE)),
        ])),
        ("forest", Pipeline(steps=[
            ("preprocess", build_preprocessor(X_reference)),
            ("classifier", RandomForestClassifier(n_estimators=160, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)),
        ])),
    ]

hard_voter = VotingClassifier(estimators=make_voting_estimators(X_train), voting="hard", n_jobs=1)
soft_voter = VotingClassifier(estimators=make_voting_estimators(X_train), voting="soft", n_jobs=1)

hard_voter.fit(X_train, y_train)
soft_voter.fit(X_train, y_train)

hard_pred = hard_voter.predict(X_test)
soft_pred = soft_voter.predict(X_test)

ensemble_results = pd.DataFrame([
    {
        "Ensemble": "Hard voting",
        "Accuracy": accuracy_score(y_test, hard_pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, hard_pred),
        "Macro F1": f1_score(y_test, hard_pred, average="macro", zero_division=0),
    },
    {
        "Ensemble": "Soft voting",
        "Accuracy": accuracy_score(y_test, soft_pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, soft_pred),
        "Macro F1": f1_score(y_test, soft_pred, average="macro", zero_division=0),
    },
])
display(ensemble_results.style.format({"Accuracy": "{:.3f}", "Balanced accuracy": "{:.3f}", "Macro F1": "{:.3f}"}))

## 25 · Final model comparison

Bring the main results together. Include the cross-validation results, final hold-out result for the selected baseline model, feature-reduction experiments, PCA and the two voting ensembles. The tables intentionally keep macro F1 and balanced accuracy visible because the dataset has an imbalanced target.

In [ ]:
final_comparison = pd.DataFrame([
    {
        "Experiment": "Best all-feature baseline (hold-out)",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
        "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    },
    {
        "Experiment": "Hard voting ensemble (hold-out)",
        "Accuracy": accuracy_score(y_test, hard_pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, hard_pred),
        "Macro F1": f1_score(y_test, hard_pred, average="macro", zero_division=0),
    },
    {
        "Experiment": "Soft voting ensemble (hold-out)",
        "Accuracy": accuracy_score(y_test, soft_pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, soft_pred),
        "Macro F1": f1_score(y_test, soft_pred, average="macro", zero_division=0),
    },
])
display(final_comparison.style.format({"Accuracy": "{:.3f}", "Balanced accuracy": "{:.3f}", "Macro F1": "{:.3f}"}))

print("Cross-validation baseline ranking:")
display(cv_results)
print("Feature reduction comparison:")
display(drop_comparison)
print("PCA comparison:")
display(pca_comparison)
print("Feature selection comparison:")
display(selection_comparison)

## 26 · Conclusions and report-writing prompts

Use the actual outputs from your notebook to answer these questions. Do not write results before running the code.

1. Did the dataset meet the professor's minimum of more than 5,000 rows and at least 10 features? State the exact numbers printed by the notebook.
2. Which class was least represented? How did the imbalance affect accuracy versus macro F1?
3. Which all-feature classifier performed best under stratified five-fold cross-validation?
4. Did aggregating and dropping component features (`dur_pt_*` and driving-cost components) improve, reduce or preserve macro F1?
5. Did `SelectKBest` improve performance while using fewer transformed features?
6. How many PCA components were needed to retain 95% of transformed variance, and how did the PCA model compare with the full-feature Logistic Regression baseline?
7. What value of *k* looked reasonable from the K-Means elbow and silhouette plots?
8. How did DBSCAN behave as `eps` changed? Report clusters and noise percentage.
9. Did hard voting or soft voting perform better on the held-out test set?
10. What are the limitations of the data and validation design?

### Limitations to acknowledge

- The data is from London and covers 2012–2015; results may not transfer directly to present-day travel in India or another city.
- Public transport is one target class. Bus and rail are route components, not separate target labels; flights are not included.
- Weather and elevation are not supplied as standard features in this dataset.
- The target is naturally imbalanced, especially for cycling.
- Multiple trips may come from the same household/person. A random stratified split may therefore be more optimistic than a grouped or time-based evaluation.
- Estimated durations/costs for alternative modes are part of the input choice-set. The prediction assumes those alternatives are available at prediction time.
- Clustering is exploratory; clusters need not match the four recorded mode labels.

### Final project scope check

This notebook uses the requested stack only: scikit-learn, NumPy, pandas, Matplotlib and Seaborn (plus Python's standard library for file-path handling). It implements the whiteboard requirements and stays within the classical methods listed in the PRML lab plan. No neural network, deep-learning, XGBoost, imbalanced-learn, or external AutoML library is required.

### References

- Hillel, T., Elshafie, M. Z. E. B., & Jin, Y. (2018). *Recreating passenger mode choice-sets for transport simulation: A case study of London, UK*. Proceedings of the Institution of Civil Engineers – Smart Infrastructure and Construction, 171(1), 29–42. [https://doi.org/10.1680/jsmic.17.00018](https://doi.org/10.1680/jsmic.17.00018)
- EPFL/Biogeme LPMC dataset description and data file: [Biogeme LPMC](https://transp-or.epfl.ch/biogeme-3.3.2/).
- Course alignment: *Pattern Recognition and Machine Learning* course outline supplied for this project, especially the module-wise course content and Lab-wise Plan (Labs 1–13).